# Flight Analytics — Silver Layer

Reads raw Bronze records (Kafka `value` as a JSON string), parses them into typed columns, applies data quality rules, deduplicates, and writes:
- valid/clean records → **Silver Delta table**
- hard-reject records → **dead-letter Delta table**

Reads Bronze as a stream (not a one-off batch), so this notebook is itself a continuously-running Structured Streaming job, chained after the Bronze write.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType, LongType,
)

print("Spark version:", spark.version)


## Paths

Same Unity Catalog Volume used for Bronze — Silver and the dead-letter table live alongside it, each with their own checkpoint (checkpoints are per-query, never shared).


In [ ]:
BASE_PATH = "/Volumes/flight_analytics/streaming/flight_data"

bronze_path = f"{BASE_PATH}/bronze"

silver_path = f"{BASE_PATH}/silver"
silver_checkpoint = f"{BASE_PATH}/checkpoints/silver"

deadletter_path = f"{BASE_PATH}/silver_deadletter"
deadletter_checkpoint = f"{BASE_PATH}/checkpoints/silver_deadletter"

print("Bronze source:", bronze_path)
print("Silver target:", silver_path)
print("Dead-letter target:", deadletter_path)


## Schema for the OpenSky JSON payload

Matches the fields published by `producer/kafka_producer.py` (parsed state vector plus the `ingestion_timestamp` the producer adds). Declaring this explicitly — instead of letting Spark infer it — means a malformed message produces nulls in `from_json`, not a job-crashing schema-inference surprise.


In [ ]:
opensky_schema = StructType([
    StructField("icao24", StringType()),
    StructField("callsign", StringType()),
    StructField("origin_country", StringType()),
    StructField("time_position", LongType()),
    StructField("last_contact", LongType()),
    StructField("longitude", DoubleType()),
    StructField("latitude", DoubleType()),
    StructField("baro_altitude", DoubleType()),
    StructField("on_ground", BooleanType()),
    StructField("velocity", DoubleType()),
    StructField("true_track", DoubleType()),
    StructField("vertical_rate", DoubleType()),
    StructField("geo_altitude", DoubleType()),
    StructField("squawk", StringType()),
    StructField("spi", BooleanType()),
    StructField("position_source", LongType()),
    StructField("category", StringType()),
    StructField("ingestion_timestamp", StringType()),
])

print("Schema defined:", len(opensky_schema.fields), "fields")


## Read Bronze as a stream and parse the JSON payload

`time_position` and `last_contact` arrive as Unix seconds (Long) — converted here to real timestamps, since `time_position` becomes our event-time column for watermarking downstream (in Gold), and dedup below relies on it too.


In [ ]:
bronze_stream = spark.readStream.format("delta").load(bronze_path)

parsed_stream = (
    bronze_stream
    .select(F.from_json(F.col("value"), opensky_schema).alias("data"))
    .select("data.*")
    .withColumn("time_position_ts", F.to_timestamp(F.col("time_position")))
    .withColumn("last_contact_ts", F.to_timestamp(F.col("last_contact")))
    .withColumn("callsign", F.trim(F.col("callsign")))
    .withColumn("callsign", F.when(F.col("callsign") == "", None).otherwise(F.col("callsign")))
)

print("Parsed stream columns:", parsed_stream.columns)


## Data quality rules

**Hard-reject rules** — a record failing any of these is not usable at all, and goes to the dead-letter table instead of Silver:
- latitude out of [-90, 90] or missing
- longitude out of [-180, 180] or missing
- negative altitude (both baro and geo, when present)
- negative velocity
- missing `icao24` (can't identify or dedup the record at all)

**Soft-flag rules** — record is kept in Silver, but flagged, since the core position/movement data is still usable:
- missing callsign
- stale position (`time_position` older than the freshness threshold relative to `ingestion_timestamp`)

A **DQ score** (percentage of checks passed) is computed per record from the soft-rule results, so pipeline health is a trackable number rather than a guess.


In [ ]:
FRESHNESS_THRESHOLD_SECONDS = 20

dq_stream = (
    parsed_stream
    # --- hard-reject checks (boolean: True = passes) ---
    .withColumn(
        "chk_icao24_present",
        F.col("icao24").isNotNull(),
    )
    .withColumn(
        "chk_lat_valid",
        F.col("latitude").isNotNull() & F.col("latitude").between(-90, 90),
    )
    .withColumn(
        "chk_lon_valid",
        F.col("longitude").isNotNull() & F.col("longitude").between(-180, 180),
    )
    .withColumn(
        "chk_altitude_valid",
        # null altitude is acceptable (aircraft can have no altitude report yet) -
        # only a NEGATIVE value is invalid
        (F.col("baro_altitude").isNull() | (F.col("baro_altitude") >= 0))
        & (F.col("geo_altitude").isNull() | (F.col("geo_altitude") >= 0)),
    )
    .withColumn(
        "chk_velocity_valid",
        F.col("velocity").isNull() | (F.col("velocity") >= 0),
    )
    # --- soft-flag checks ---
    .withColumn(
        "chk_callsign_present",
        F.col("callsign").isNotNull(),
    )
    .withColumn(
        "chk_position_fresh",
        F.col("time_position_ts").isNotNull()
        & (
            F.unix_timestamp(F.to_timestamp(F.col("ingestion_timestamp")))
            - F.unix_timestamp(F.col("time_position_ts"))
            <= FRESHNESS_THRESHOLD_SECONDS
        ),
    )
    # --- combined outcomes ---
    .withColumn(
        "is_hard_valid",
        F.col("chk_icao24_present")
        & F.col("chk_lat_valid")
        & F.col("chk_lon_valid")
        & F.col("chk_altitude_valid")
        & F.col("chk_velocity_valid"),
    )
    .withColumn(
        "dq_score",
        (
            F.col("chk_callsign_present").cast("int")
            + F.col("chk_position_fresh").cast("int")
            + F.col("chk_icao24_present").cast("int")
            + F.col("chk_lat_valid").cast("int")
            + F.col("chk_lon_valid").cast("int")
            + F.col("chk_altitude_valid").cast("int")
            + F.col("chk_velocity_valid").cast("int")
        )
        / F.lit(7.0)
        * 100,
    )
)

print("DQ columns added.")


## Deduplication

`dropDuplicatesWithinWatermark` (Spark 3.5+) dedupes on `(icao24, time_position)` within the watermark window — same aircraft + same timestamp = duplicate; same aircraft + different timestamp = a genuine new observation, kept.

The watermark itself (30s) bounds how long Spark keeps state around for dedup, matching the same tolerance used for windowed aggregation later in Gold.


In [ ]:
deduped_stream = (
    dq_stream
    .withWatermark("time_position_ts", "30 seconds")
    .dropDuplicatesWithinWatermark(["icao24", "time_position"])
)

print("Dedup configured on (icao24, time_position) with a 30s watermark.")


## Write: split each micro-batch into Silver (valid) and dead-letter (hard-reject)

`foreachBatch` lets one streaming query write to two different Delta tables per micro-batch, using `MERGE` for idempotent upserts — safe to re-run without creating duplicate rows if the job restarts mid-batch.


In [ ]:
from delta.tables import DeltaTable

def write_silver_and_deadletter(batch_df, batch_id):
    # NOTE: no .persist() here - Databricks Serverless compute does not support
    # manual DataFrame caching (PERSIST TABLE is disallowed on serverless).
    # The batch_df is small per micro-batch, so recomputation cost is negligible.

    valid_df = batch_df.filter(F.col("is_hard_valid"))
    invalid_df = batch_df.filter(~F.col("is_hard_valid"))

    # --- Silver: idempotent merge on (icao24, time_position) ---
    if DeltaTable.isDeltaTable(spark, silver_path):
        silver_table = DeltaTable.forPath(spark, silver_path)
        (
            silver_table.alias("tgt")
            .merge(
                valid_df.alias("src"),
                "tgt.icao24 = src.icao24 AND tgt.time_position = src.time_position",
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        valid_df.write.format("delta").mode("overwrite").save(silver_path)

    # --- Dead-letter: append-only, every hard-reject record kept for debugging ---
    invalid_df.write.format("delta").mode("append").save(deadletter_path)

    batch_valid_count = valid_df.count()
    batch_invalid_count = invalid_df.count()
    batch_total = batch_valid_count + batch_invalid_count
    batch_dq_pct = (batch_valid_count / batch_total * 100) if batch_total else 100.0

    print(
        f"Batch {batch_id}: {batch_valid_count} valid, {batch_invalid_count} rejected "
        f"({batch_dq_pct:.1f}% batch DQ)"
    )

print("foreachBatch writer defined.")


In [ ]:
silver_query = (
    deduped_stream
    .writeStream
    .foreachBatch(write_silver_and_deadletter)
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .start()
)

silver_query.awaitTermination()
print("Silver write finished.")


## Sanity checks


In [ ]:
print("--- Silver sample ---")
display(
    spark.read.format("delta").load(silver_path)
    .orderBy(F.col("time_position_ts").desc())
    .limit(20)
)


In [ ]:
print("--- Dead-letter sample (should show WHY each record failed) ---")
display(
    spark.read.format("delta").load(deadletter_path)
    .select(
        "icao24", "callsign", "latitude", "longitude", "baro_altitude", "velocity",
        "chk_icao24_present", "chk_lat_valid", "chk_lon_valid",
        "chk_altitude_valid", "chk_velocity_valid",
    )
    .limit(20)
)


In [ ]:
silver_count = spark.read.format("delta").load(silver_path).count()
deadletter_count = spark.read.format("delta").load(deadletter_path).count()
total = silver_count + deadletter_count
overall_dq_pct = (silver_count / total * 100) if total else 0

print(f"Silver rows: {silver_count}")
print(f"Dead-letter rows: {deadletter_count}")
print(f"Overall DQ score: {overall_dq_pct:.2f}%")
